# Track 2 · Stage 1 — Generate the code-mixed **text**

Replication of Biswas et al., Interspeech 2025 (Track 2).

Few-shot-prompt **Gemma 4 E4B** for Hindi-English bigrams → filter them → expand each
into four sentences (~16k). Push the text to `RohanRamesh/hi-en-synth-cs`.

Audio synthesis is a **separate notebook** (`01b`). `parler-tts` hard-pins
`transformers==4.46.1`, Gemma 4 needs `transformers>=5.5` — they cannot coexist in one
process. The Hub is the checkpoint between them.

### The model: `google/gemma-4-E4B-it`
* **apache-2.0, ungated** — no licence click-through, no gated-repo token scope, no mirror
  needed. (Gemma 3's `google/*` repos *are* gated; Gemma 4's are not.)
* Deviation **D1**: the paper used Llama-3.3-70B (141 GB bf16). Gemma 4 E4B is ~8B raw
  params, pretrained on 140+ languages, and a **deterministic script filter** sits behind it
  to catch what it still gets wrong.
* The one **live risk**: Gemma 4 is **bf16-native** (`torch_dtype: bfloat16`) and the T4 is
  Turing — it has **no bf16**. fp16 activations can exceed 65,504 and go non-finite.
  `backend.py` probes the logits after loading and transparently reloads with float32 compute
  if they are NaN/inf. **The smoke test below is how you find out**, in about a minute.
* Two non-issues, confirmed against the real config (the backend defends against both anyway):
  it is `Gemma4ForConditionalGeneration` but *is* registered under `AutoModelForCausalLM`, and
  its chat template **does** support a `system` role — so the paper's verbatim system prompt
  survives intact.

### Before you run
* HF **write** token in Kaggle Secrets as `HF_TOKEN`. Internet **on**, **GPU T4 ×2**.
* Nothing to accept — Gemma 4 is apache-2.0. (Parler-TTS *is* gated; that bites in `01b`.)

Runtime ≈ 2h. Every LLM call is cached, so a 12h timeout costs nothing on a re-run.

## 0 · Install

In [ ]:
# Gemma 4 needs transformers >= 5.5. NO parler-tts here -- that is 01b's problem.
!pip install -q -U "transformers>=5.5" accelerate bitsandbytes
!pip install -q "datasets<4" librosa soundfile soxr omegaconf rich
!pip install -q --force-reinstall --no-deps git+https://github.com/BRUH-MAIN/codeswitching.git

import csasr, transformers
assert csasr.__version__ >= "0.7.0", (
    f"stale csasr {csasr.__version__} - the kernel is running old code. "
    "Restart the kernel (Run > Restart & clear) and re-run this cell."
)
from packaging.version import Version
assert Version(transformers.__version__) >= Version("5.5"), (
    f"transformers {transformers.__version__} cannot load Gemma 4 (needs >= 5.5)."
)
print("csasr", csasr.__version__, "| transformers", transformers.__version__)

In [ ]:
import os, subprocess, sys
from pathlib import Path

os.environ["HF_HOME"] = "/kaggle/temp/hf"
Path(os.environ["HF_HOME"]).mkdir(parents=True, exist_ok=True)

from kaggle_secrets import UserSecretsClient
os.environ["HF_TOKEN"] = UserSecretsClient().get_secret("HF_TOKEN")
from huggingface_hub import login
login(token=os.environ["HF_TOKEN"])
HF_TOKEN = os.environ["HF_TOKEN"]

REAL_REPO  = "RohanRamesh/mucs-he-cs"
SYNTH_REPO = "RohanRamesh/hi-en-synth-cs"
LLM        = "google/gemma-4-E4B-it"      # apache-2.0, ungated. E2B is the smaller option.

WORK  = Path("/kaggle/working")
MAN   = WORK / "manifests"; MAN.mkdir(parents=True, exist_ok=True)
CACHE = WORK / "llm_cache"; CACHE.mkdir(parents=True, exist_ok=True)

def run(*args):
    print(">", " ".join(str(a) for a in args), flush=True)
    p = subprocess.run([sys.executable, "-m", *args])   # inherits HF_TOKEN + streams
    if p.returncode != 0:
        raise RuntimeError(
            f"{args[0]} failed (exit {p.returncode}). The real error is printed ABOVE "
            f"this traceback - scroll up in this cell's output."
        )

!nvidia-smi --query-gpu=name,memory.total --format=csv

## 1 · Pull the in-domain transcripts (few-shot exemplars)

Track 2 never trains on real code-switched audio — we only need the *text* of the MUCS train split.

In [ ]:
from datasets import load_dataset
from csasr.manifest import write_jsonl

train_text = load_dataset(REAL_REPO, "train_text", split="train", token=HF_TOKEN)
write_jsonl(MAN / "mucs_train.jsonl", [dict(r) for r in train_text])
print(f"{len(train_text):,} in-domain sentences for few-shot prompting")
print(train_text[0]["text"])

## 2 · SMOKE TEST — 20 bigrams before committing 2 hours

Loads Gemma, runs the **fp16 → float32 logits health check** (Gemma 4 is bf16-native and
the T4 has no bf16), generates real bigrams, and shows which survive the script filter.

**It runs as a subprocess, deliberately.** Jupyter's `Out[]` history holds a reference to
anything a cell produced, so `del model` does *not* free the VRAM — and the next subprocess
then dies with `Some modules are dispatched on the CPU or the disk`. Keeping the model out
of the kernel entirely is the only reliable fix.

Expect Gemma to emit **three**-word phrases (`बुनियादी formatting basics`). That is fine: the
filter extracts the switch pair from inside them. See deviation **D9**.

In [ ]:
run("csasr.llm.smoke", "--model", LLM,
    "--train-manifest", MAN / "mucs_train.jsonl",
    "--n-calls", "2", "--bigrams-per-call", "10")

## 3 · Generate bigrams

Paper: 44,657 raw → 5,932 unique (13.3%).

In [ ]:
run("csasr.llm.gen_bigrams",
    "--train-manifest", MAN / "mucs_train.jsonl",
    "--out", MAN / "bigrams_raw.jsonl",
    "--cache", CACHE / "bigrams.jsonl",
    "--model", LLM, "--n-calls", "4466", "--batch-size", "16")

## 4 · Filter

Deterministic script filter (one Devanagari token + one Latin token), then an LLM translation check with 3-sample self-consistency.
Paper: 5,932 unique → 5,477 valid (92.3%).

In [ ]:
run("csasr.llm.filter_bigrams",
    "--raw", MAN / "bigrams_raw.jsonl",
    "--out", MAN / "bigrams_valid.jsonl",
    "--cache", CACHE / "transcheck.jsonl",
    "--model", LLM, "--items-per-call", "20", "--n-samples", "3")

## 5 · Expand each bigram into four sentences

2 English-matrix, 2 Hindi-matrix. Paper: ~16,000 unique from a theoretical 21,908.

In [ ]:
run("csasr.llm.gen_sentences",
    "--bigrams", MAN / "bigrams_valid.jsonl",
    "--out", MAN / "sentences.jsonl",
    "--cache", CACHE / "sentences.jsonl",
    "--model", LLM, "--batch-size", "16")

### GATE 1 — yields must track the paper

A large divergence in the 13.3% dedup rate means the prompt or temperature is off. Decide here, not after 6 hours of TTS.

In [ ]:
from csasr.manifest import read_jsonl

raw   = list(read_jsonl(MAN / "bigrams_raw.jsonl"))
uniq  = {r["bigram"] for r in raw}
valid = list(read_jsonl(MAN / "bigrams_valid.jsonl"))
sents = list(read_jsonl(MAN / "sentences.jsonl"))

rows = [
    ("raw bigrams",    len(raw),   44_657, None),
    ("unique bigrams", len(uniq),   5_932, len(uniq) / max(len(raw), 1)),
    ("valid bigrams",  len(valid),  5_477, len(valid) / max(len(uniq), 1)),
    ("sentences",      len(sents), 16_000, None),
]
print(f"{'metric':<16}{'ours':>10}{'paper':>10}{'survival':>12}")
for name, got, want, surv in rows:
    s = f"{surv:.1%}" if surv else "-"
    print(f"{name:<16}{got:>10,}{want:>10,}{s:>12}")
print("\npaper survival: dedup 13.3%, filter 92.3%")
print("\nGemma 3 4B is far smaller than the paper's 70B (deviation D1), so a lower")
print("valid-bigram yield is expected. What matters is that ENOUGH sentences survive:")
print(f"  -> {len(sents):,} sentences  (need >~8,000 for a usable 22h corpus)")

for r in sents[:5]:
    print("   ", r["text"])

## 6 · Push the text to the Hub

This is the handoff to `01b`. Push before anything can time out.

In [ ]:
for man, cfg in [("bigrams_valid.jsonl", "bigrams"), ("sentences.jsonl", "sentences")]:
    run("csasr.data.push_to_hub", "--manifest", MAN / man,
        "--repo", SYNTH_REPO, "--config", cfg, "--text-only")
print("\ntext stage complete -> now run 01b_synthesize_audio.ipynb")